# Right-censored fitted distribution API

This notebook shows the new package contract: `fit()` still returns the estimator, and `get_fit()` returns a lightweight fitted distribution with frozen normalization constants.

```python
estimator.fit(data)
fitted = estimator.get_fit()

fitted.density(t)
fitted.survival(t)
fitted.rmst(tau)
fitted.beyond_last_failure(t)
fitted.to_state() / RightCensoredHALFit.from_state(...)
```

Queries do not recompute `Z`. Existing `get_density_at_points()` still works and matches `fitted.density`. There is no warning after the last failure; that is a diagnostic flag only.

Data: Beta event times on `[0, 1]` with independent `Unif[0, 1.2]` censoring.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from haldensity.censoring.right import (
    KaplanMeier,
    RightCensoredHALFit,
    RightCensoredInitEstimator,
    RightCensoredObservedL1MLE,
    compute_ipcw_weights,
)
from haldensity.censoring.right.comparison import simulate_beta_uniform_right_censored

## Simulate observed data

In [ ]:
sim = simulate_beta_uniform_right_censored(n=200, seed=7)
data = sim["observed_data"]
truth = sim["truth"]

n = len(data)
n_event = int(data["Delta"].sum())
print(f"n={n}, events={n_event}, censored={n - n_event}, last failure={float(data.loc[data['Delta'] == 1, 'T'].max()):.4f}")
data.head()

## Fit IPCW, then take `get_fit()`

`fit()` returns the estimator. The fitted distribution is a separate object.

In [ ]:
km = KaplanMeier().fit(data, time_col="T", delta_col="Delta")
t_vals = data["T"].to_numpy(dtype=float)
delta_vals = data["Delta"].to_numpy(dtype=int)
weights = compute_ipcw_weights(t_vals, delta_vals, lambda t: np.atleast_1d(km.predict(t)))

ipcw = RightCensoredInitEstimator(
    norm_constraint=8.0,
    n_grid_points=80,
    basis_order=0,
    include_intercept_in_constraint=False,
    use_secondary_solver=True,
)
returned = ipcw.fit(
    pd.DataFrame({"W1": t_vals[delta_vals == 1]}),
    sample_weights=weights[delta_vals == 1],
)
assert returned is ipcw

fitted = ipcw.get_fit()
print(type(fitted).__name__)
print("basis_order", fitted.basis_order, "n_cells", fitted.n_cells, "method", fitted.normalizer_method)
print("last_failure", fitted.last_failure)

## Query density, survival, and RMST

`get_density_at_points` still exists and must match `fitted.density`.

In [ ]:
t0 = np.array([0.25, 0.50, 0.75])
tau = 0.75

density = fitted.density(t0)
survival = fitted.survival(t0)
rmst = fitted.rmst(tau)

summary = pd.DataFrame({
    "t": t0,
    "fitted.density": density,
    "get_density_at_points": ipcw.get_density_at_points(t0),
    "fitted.survival": survival,
    "truth.survival": truth.survival_fn(t0),
    "beyond_last_failure": fitted.beyond_last_failure(t0),
})
print(f"RMST({tau}) = {float(np.asarray(rmst).reshape(-1)[0]):.4f}")
print("max |density API difference| =", np.max(np.abs(summary["fitted.density"] - summary["get_density_at_points"])))
summary

In [ ]:
grid = np.linspace(0.01, 0.99, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(grid, truth.density_fn(grid), color="black", lw=2, label="truth")
axes[0].plot(grid, fitted.density(grid), lw=2, label="fitted.density")
axes[0].set_title("Density")
axes[0].set_xlabel("t")
axes[0].legend()

axes[1].plot(grid, truth.survival_fn(grid), color="black", lw=2, label="truth")
axes[1].plot(grid, fitted.survival(grid), lw=2, label="fitted.survival")
if fitted.last_failure is not None:
    axes[1].axvline(fitted.last_failure, color="gray", ls="--", label="last failure")
axes[1].set_title("Survival")
axes[1].set_xlabel("t")
axes[1].legend()
fig.tight_layout()
plt.show()

## After the last failure

Still return the number. No warning. `beyond_last_failure` is a diagnostic.

In [ ]:
import warnings

query = np.array([fitted.last_failure, min(fitted.last_failure + 0.05, 1.0), 1.0])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    tail = pd.DataFrame({
        "t": query,
        "density": fitted.density(query),
        "survival": fitted.survival(query),
        "beyond_last_failure": fitted.beyond_last_failure(query),
    })
print("warnings emitted:", len(caught))
tail

## Save and reload without rebuilding `Z`

In [ ]:
reloaded = RightCensoredHALFit.from_state(fitted.to_state())
check = np.linspace(0.05, 0.95, 21)
print("density reload max abs err", np.max(np.abs(reloaded.density(check) - fitted.density(check))))
print("survival reload max abs err", np.max(np.abs(reloaded.survival(check) - fitted.survival(check))))
print("rmst reload abs err", float(np.abs(reloaded.rmst(0.75) - fitted.rmst(0.75))))
print("same last_failure", reloaded.last_failure == fitted.last_failure)

## L1MLE uses the same object

Stage 2 inherits the IPCW-selected knots. `get_fit()` is the same API.

In [ ]:
support = np.asarray(ipcw.grid_points_hal_selected, dtype=float)
all_knots = np.asarray(ipcw._grid_points_hal, dtype=float)
theta_full = np.asarray(ipcw.theta_hat, dtype=float)
theta_selected = np.zeros(1 + support.size, dtype=float)
theta_selected[0] = theta_full[0]
for i, knot in enumerate(support):
    idx = np.where(np.isclose(all_knots, knot, atol=1e-10, rtol=0.0))[0][0]
    theta_selected[1 + i] = theta_full[1 + idx]

l1mle = RightCensoredObservedL1MLE(
    working_grid_points=support,
    norm_constraint=float(ipcw.norm_constraint),
    basis_order=0,
    n_grid_points=80,
    learning_rate=0.1,
    n_iterations=150,
    ll_change_tol=1e-4,
    include_intercept_in_constraint=False,
    warm_start_theta=theta_selected,
)
assert l1mle.fit(data) is l1mle
l1_fit = l1mle.get_fit()

print("L1MLE last_failure", l1_fit.last_failure)
print("L1MLE RMST(0.75)", float(np.asarray(l1_fit.rmst(0.75)).reshape(-1)[0]))
print(
    "L1MLE density matches get_density_at_points",
    np.allclose(l1_fit.density(t0), l1mle.get_density_at_points(t0)),
)